In [ ]:
import os
import pandas as pd
from ultralytics import YOLO

# --- Konfigurasi Proyek ---
project_path = r"C:\AI\makanan\Hasil"  # Lokasi folder proyek
train_name = "yolo11s-seg(v8)"  # Nama model pelatihan sebelumnya
results_path = f"{project_path}/{train_name}/results.csv"  # Lokasi file hasil pelatihan
model_path = f"{project_path}/{train_name}/weights/best.pt"  # Lokasi model terbaik yang disimpan

# --- Memeriksa Hasil Training Sebelumnya ---
if os.path.exists(results_path):
    df = pd.read_csv(results_path)  # Membaca file CSV hasil pelatihan

    # Cek loss terakhir
    if "val/mask_loss" in df.columns:
        last_loss = df["val/mask_loss"].iloc[-1]
        loss_type = "val/mask_loss"
    else:
        last_loss = df["val/box_loss"].iloc[-1]
        loss_type = "val/box_loss"
    
    print(f"Loss terakhir ({loss_type}): {last_loss:.4f}")

    # Cek akurasi terakhir
    if "metrics/mAP50-95(M)" in df.columns:
        last_accuracy = df["metrics/mAP50-95(M)"].iloc[-1]
        print(f"Akurasi terakhir (mAP50-95(M)): {last_accuracy:.2f}")
    else:
        last_accuracy = None
        print("Kolom akurasi (metrics/mAP50-95(M)) tidak ditemukan dalam results.csv.")

    # Cek class loss
    if "val/cls_loss" in df.columns:
        cls_loss = df["val/cls_loss"].iloc[-1]
        print(f"Class Loss terakhir (val/cls_loss): {cls_loss:.4f}")
    else:
        cls_loss = 0.0
        print("Kolom class loss (val/cls_loss) tidak ditemukan dalam results.csv.")

    # --- Kondisi untuk Melanjutkan Training ---
    if last_loss > 0.10 or (last_accuracy is not None and last_accuracy < 95) or cls_loss > 0.05:
        print("Melanjutkan training dengan optimasi...")

        # Load Model
        model = YOLO(model_path)

        # Train ulang dengan setting optimasi
        model.train(
            data=r"C:\AI\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6\dataset.yaml",
            device="cuda",
            project=project_path,
            name="yolo11s-seg(v9)",

            # Core Training
            epochs=300,
            batch=16,
            imgsz=896,
            workers=4,
            optimizer="AdamW",
            lr0=0.00000000000001,
            lrf=0.00000000001,
            momentum=0.95,
            weight_decay=0.000001,
            cos_lr=True,
            warmup_epochs=5,
            patience=5,

            # Augmentasi khusus segmentasi tumpuk
            degrees=5,
            translate=0.05,
            scale=0.3,
            shear=1,
            perspective=0.0005,
            flipud=0.05,
            fliplr=0.5,
            hsv_h=0.1,     # ubah warna hue lebih kuat
            hsv_s=0.8,      # saturasi lebih liar
            hsv_v=0.6,      # brightness bervariasi
            mixup=0.0,
            mosaic=0.05,
            bgr=0.1,
            copy_paste=0.3,

            # Tambahan fitur
            multi_scale=False,
            # label_smoothing=0.001,
            cache='disk',
            overlap_mask=True,
            save_crop=True,
            plots=True,
            save=True,
            verbose=True,
            visualize=True
        )
        print("Training tambahan selesai!")
    else:
        print("Loss cukup rendah dan akurasi sudah bagus, training tidak dilanjutkan.")
else:
    print("File results.csv tidak ditemukan. Pastikan training sebelumnya sudah berjalan.")

Loss terakhir (val/box_loss): 0.3215
Akurasi terakhir (mAP50-95(M)): 0.87
Class Loss terakhir (val/cls_loss): 0.2243
Melanjutkan training dengan optimasi...
Ultralytics 8.3.119  Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: task=segment, mode=train, model=C:\AI\makanan\Hasil/yolo11s-seg(v8)/weights/best.pt, data=C:\AI\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6\dataset.yaml, epochs=300, time=None, patience=5, batch=16, imgsz=896, save=True, save_period=-1, cache=disk, device=cuda, workers=4, project=C:\AI\makanan\Hasil, name=yolo11s-seg(v9), exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=

train: Scanning C:\AI\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6\labels\train... 4318 images, 0 backgrounds, 0 corrupt: 100%|██████████| 4318/4318 [00:09<00:00, 441.29it/s]

train: C:\AI\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6\images\train\tempe (435)_aug_4.jpg: 1 duplicate labels removed
train: C:\AI\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6\images\train\tempe (435)_aug_6.jpg: 1 duplicate labels removed


train: New cache created: C:\AI\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6\labels\train.cache


train: Caching images (11.1GB Disk): 100%|██████████| 4318/4318 [00:12<00:00, 348.05it/s]


...

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validati

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95):   7%|▋         | 2/28 [00:01<00:21,  1.23it/s]

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validati

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 28/28 [00:14<00:00,  1.89it/s]


                   all        896       7970      0.978      0.961      0.988      0.928      0.977      0.965      0.986      0.879
                bawang        600       3503      0.989      0.987      0.994      0.949      0.987      0.988      0.994      0.901
                 tempe        446       4467      0.968      0.935      0.982      0.908      0.967      0.942      0.978      0.857
Speed: 0.4ms preprocess, 6.2ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\AI\makanan\Hasil\yolo11s-seg(v9)
Training tambahan selesai!


In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2

# Load the model
model = YOLO(r"D:\MyProjects\makanan\Hasil\yolo11s-seg(v6)\weights\best.pt")

# Predict with the model
results = model(r"D:\MyProjects\makanan\Dataset_workshop\Gambar_awal\Tempe\tempe\tempe (332).jpg")

# Visualize the results
for result in results:
    # Plot the result (with masks, bounding boxes, labels)
    img_with_result = result.plot()

    # Convert BGR to RGB for matplotlib
    img_rgb = cv2.cvtColor(img_with_result, cv2.COLOR_BGR2RGB)

    # Show image using matplotlib
    plt.figure(figsize=(10, 10))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title("Prediction Result")
    plt.show()

    cv2.imwrite("hasil_prediksi.jpg", img_with_result)

In [1]:
import os
from ultralytics import YOLO

# =============================
# Konfigurasi Proyek
# =============================
project_path = r"C:\AI\kue\Hasil"
model_path = r"C:\AI\kue\Base_model\yolo11n-seg.pt"
data_yaml_path = r"C:\AI\kue\Dataset_final\Dataset_yolo_Segmentation_v1\dataset.yaml"

# =============================
# Training Model YOLO
# =============================
model = YOLO(model_path)

model.train(
    data=data_yaml_path,
    device="cuda",
    project=project_path,
    name="yolo11n-seg(v1)",

    # Core Training
    epochs=500,
    batch=18,
    imgsz=896,
    workers=5,
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.01,
    momentum=0.8,
    weight_decay=1e-6,
    cos_lr=True,
    warmup_epochs=10,
    patience=20,

    # Augmentasi
    degrees=5,
    translate=0.05,
    scale=0.3,
    shear=1,
    perspective=0.0005,
    flipud=0.05,
    fliplr=0.5,
    hsv_h=0.1,
    hsv_s=0.8,
    hsv_v=0.6,
    mixup=0.0,
    mosaic=0.05,
    copy_paste=0.3,

    # Fitur Tambahan
    multi_scale=True,
    cache='disk',
    overlap_mask=True,
)

print("Training selesai!")

Ultralytics 8.3.120  Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: task=segment, mode=train, model=C:\AI\kue\Base_model\yolo11n-seg.pt, data=C:\AI\kue\Dataset_final\Dataset_yolo_Segmentation_v1\dataset.yaml, epochs=500, time=None, patience=20, batch=18, imgsz=896, save=True, save_period=-1, cache=disk, device=cuda, workers=5, project=C:\AI\kue\Hasil, name=yolo11n-seg(v1), exist_ok=False, pretrained=True, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=True, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=True, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, 

train: Scanning C:\AI\kue\Dataset_final\Dataset_yolo_Segmentation_v1\labels\train.cache... 295 images, 0 backgrounds, 0 corrupt: 100%|██████████| 295/295 [00:00<?, ?it/s]
train: Caching images (0.6GB Disk): 100%|██████████| 295/295 [00:00<00:00, 46402.39it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access  (ping: 0.00.0 ms, read: 1161.2602.7 MB/s, size: 439.1 KB)


val: Scanning C:\AI\kue\Dataset_final\Dataset_yolo_Segmentation_v1\labels\val.cache... 59 images, 0 backgrounds, 0 corrupt: 100%|██████████| 59/59 [00:00<?, ?it/s]
val: Caching images (0.3GB Disk): 100%|██████████| 59/59 [00:00<00:00, 58571.35it/s]


Plotting labels to C:\AI\kue\Hasil\yolo11n-seg(v1)\labels.jpg... 
optimizer: AdamW(lr=0.001, momentum=0.8) with parameter groups 90 weight(decay=0.0), 101 weight(decay=1.125e-06), 100 bias(decay=0.0)
TensorBoard: model graph visualization added 
Image sizes 896 train, 896 val
Using 5 dataloader workers
Logging results to C:\AI\kue\Hasil\yolo11n-seg(v1)
Starting training for 500 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size


...


166 epochs completed in 4.000 hours.
Optimizer stripped from C:\AI\kue\Hasil\yolo11n-seg(v1)\weights\last.pt, 6.0MB
Optimizer stripped from C:\AI\kue\Hasil\yolo11n-seg(v1)\weights\best.pt, 6.0MB

Validating C:\AI\kue\Hasil\yolo11n-seg(v1)\weights\best.pt...
Ultralytics 8.3.120  Python-3.12.7 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
YOLO11n-seg summary (fused): 113 layers, 2,835,543 parameters, 0 gradients, 10.2 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95):   0%|          | 0/2 [00:00<?, ?it/s]

WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...
WARNING Limiting validation plots to first 50 items per image for speed...


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100%|██████████| 2/2 [00:01<00:00,  1.98it/s]


                   all         59        589      0.995      0.995      0.995      0.968      0.995      0.995      0.995      0.909
                lemper         58        116      0.991      0.982      0.994      0.946      0.991      0.982      0.994      0.866
                pastel         58        116      0.991      0.996      0.995      0.987      0.991      0.996      0.995      0.952
             kue lapis         58        116      0.999          1      0.995      0.971      0.999          1      0.995      0.911
           kue mangkok         58        113      0.991          1      0.995       0.96      0.991          1      0.995      0.911
                 wajik         59        128          1      0.996      0.995      0.975          1      0.996      0.995      0.905
Speed: 1.2ms preprocess, 5.3ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to C:\AI\kue\Hasil\yolo11n-seg(v1)
Training selesai!


In [9]:
import torch
from ultralytics import YOLO

# Load full model dari hasil training yang belum selesai
full_model_path = r"D:\MyProjects\makanan\Hasil\yolo11s-obb(v1)\weights\best.pt"
model = YOLO(full_model_path)

# Ekstrak hanya bagian yang dibutuhkan untuk inference
model_frozen = model.model

# Simpan ulang model ke versi kecil (tanpa optimizer, tanpa training info)
compressed_model_path = "eksport/last_clean.pt"
torch.save({'model': model_frozen}, compressed_model_path)

print(f"Model cleaned & saved at {compressed_model_path}")

Model cleaned & saved at eksport/last_clean.pt


In [ ]:
from ultralytics import YOLO

# Load model hasil training
model = YOLO("path/to/best.pt")
img_size = 896  # sesuai dengan saat training

# ==== 1. Ekspor ke TorchScript ====
model.export(format="torchscript", imgsz=img_size, optimize=True)

# ==== 2. Ekspor ke ONNX ====
model.export(format="onnx", imgsz=img_size, simplify=True)

# ==== 3. Ekspor ke ONNX (dynamic input size) ====
model.export(format="onnx", imgsz=img_size, dynamic=True, simplify=True)

# ==== 4. ONNX dengan FP16 (kalau hardware support) ====
model.export(format="onnx", imgsz=img_size, half=True, simplify=True)

# ==== 5. TensorRT (engine) ====
model.export(format="engine", imgsz=img_size)

# ==== 6. TensorRT INT8 (butuh data.yaml) ====
model.export(format="engine", imgsz=img_size, int8=True, data="path/to/data.yaml")

# ==== 7. OpenVINO ====
model.export(format="openvino", imgsz=img_size)

# ==== 8. CoreML (iOS/macOS) ====
model.export(format="coreml", imgsz=img_size)

# ==== 9. TensorFlow SavedModel ====
model.export(format="saved_model", imgsz=img_size)

# ==== 10. TFLite (Android) ====
model.export(format="tflite", imgsz=img_size)

# ==== 11. TF.js (web) ====
model.export(format="tfjs", imgsz=img_size)

# ==== 12. Paddle ====
model.export(format="paddle", imgsz=img_size)

# ==== 13. NCNN (mobile C++ runtime) ====
model.export(format="ncnn", imgsz=img_size)

# ==== 14. MNN (Alibaba mobile framework) ====
model.export(format="mnn", imgsz=img_size)

In [ ]:
import os

# Path ke folder label
label_dir = r"D:\MyProjects\makanan\Dataset_workshop\other\data\tempe_label"

# Proses semua file .txt dalam folder
for filename in os.listdir(label_dir):
    if filename.endswith('.txt'):
        file_path = os.path.join(label_dir, filename)
        
        with open(file_path, 'r') as f:
            lines = f.readlines()
        
        updated_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 0 and parts[0] == '0':
                parts[0] = '1'
            updated_line = ' '.join(parts)
            updated_lines.append(updated_line)
        
        # Overwrite file asli
        with open(file_path, 'w') as f:
            f.write('\n'.join(updated_lines))

        print(f"Updated: {file_path}")

print("Selesai update semua label.")

In [5]:
import os
import cv2
import numpy as np
import albumentations as A
from albumentations import ReplayCompose
from glob import glob
from tqdm import tqdm
from natsort import natsorted  # Untuk natural sort

# =============================================================================
# KONFIGURASI DAN PATH
# =============================================================================

BASE_PATH = r"C:\AI\makanan\Dataset_workshop\other\data"
IMAGE_PATH = os.path.join(BASE_PATH, "tempe2")
LABEL_PATH = os.path.join(BASE_PATH, "label")
OUTPUT_IMAGE_PATH = os.path.join(BASE_PATH, "augmented_tempe")
OUTPUT_LABEL_PATH = os.path.join(BASE_PATH, "augmented_tempe")
os.makedirs(OUTPUT_IMAGE_PATH, exist_ok=True)
os.makedirs(OUTPUT_LABEL_PATH, exist_ok=True)

# SELECT_IMAGE_START = "Bawang (431)"   # Nama dasar gambar awal (tanpa ekstensi)
# SELECT_IMAGE_END = "Bawang (450)"       # Nama dasar gambar akhir (tanpa ekstensi)
SELECT_IMAGE_START = "tempe (1)"  
SELECT_IMAGE_END = "tempe (186)"      
NUM_AUGMENTATIONS_PER_IMAGE = 15
COPY_ORIGINAL_ENABLED = True

# Variabel Global untuk strategi multi-scale/tiling dan padding
MULTISCALE_ENABLED = True
REFLECTED_PADDING_ENABLED = True
TARGET_SIZE = 869  # Resolusi output standar

# =============================================================================
# FUNGSI BANTUAN
# =============================================================================

def polygon_to_mask(img_shape, polygon):
    """
    Membuat mask biner berdasarkan poligon.
    Args:
        img_shape: Tuple (tinggi, lebar, channel) gambar.
        polygon: Array koordinat poligon (N, 2).
    Returns:
        Mask biner dengan ukuran gambar.
    """
    mask = np.zeros(img_shape[:2], dtype=np.uint8)
    pts = polygon.reshape((-1, 1, 2)).astype(np.int32)
    cv2.fillPoly(mask, [pts], 255)
    return mask

def ensure_clockwise(pts):
    """
    Pastikan titik poligon berurutan searah jarum jam.
    Jika area (metode Shoelace) positif (artinya berurutan counter-clockwise), balik urutannya.
    """
    area = 0
    for i in range(len(pts)):
        j = (i + 1) % len(pts)
        area += pts[i][0] * pts[j][1] - pts[j][0] * pts[i][1]
    if area > 0:
        pts = pts[::-1]
    return pts

def remove_duplicate_labels(label_data, precision=4):
    """
    Menghapus duplikasi label berdasarkan class_id dan koordinat poligon.
    Label dianggap duplikat jika class_id-nya sama dan koordinat poligon (setelah pembulatan)
    identik.
    """
    unique = {}
    for class_id, pts in label_data:
        # Bulatkan koordinat untuk menghindari perbedaan kecil
        rounded = tuple(np.round(pts.flatten(), precision))
        key = (class_id, rounded)
        unique[key] = (class_id, pts)
    return list(unique.values())

# =============================================================================
# PIPELINE AUGMENTASI (ReplayCompose)
# =============================================================================

# Tentukan border_mode berdasarkan opsi padding
border_mode = cv2.BORDER_REFLECT if REFLECTED_PADDING_ENABLED else cv2.BORDER_CONSTANT

# Jika multi-scale diaktifkan, tambahkan transformasi skala acak
scale_transforms = []
if MULTISCALE_ENABLED:
    scale_transforms.append(A.RandomScale(scale_limit=(0.5, 1.5), p=1.0))

# Buat pipeline dengan transformasi yang sudah digabungkan
# transform_list = scale_transforms + [
#     A.HorizontalFlip(p=0.9),
#     A.VerticalFlip(p=0.9),
#     A.RandomBrightnessContrast(brightness_limit=(0.1, 0.3), contrast_limit=(0.1, 0.3), p=0.3),
#     A.HueSaturationValue(hue_shift_limit=(5, 10), sat_shift_limit=(5, 20), val_shift_limit=(5, 30), p=0.4),
#     A.Affine(scale=(0.5, 1.5), translate_percent=(0.01, 0.2), shear=(0.1, 0.5), p=0.8),
#     A.Rotate(limit=(1, 360), p=1.0),
#     A.GaussNoise(std_range=(0.1, 0.3), p=0.3),
#     A.MotionBlur(blur_limit=(3, 9), p=0.3),
#     A.Sharpen(alpha=(0.1, 0.9), p=0.3),
#     A.LongestMaxSize(max_size=TARGET_SIZE),
#     A.PadIfNeeded(min_height=TARGET_SIZE, min_width=TARGET_SIZE, border_mode=border_mode)
# ]

transform_list = scale_transforms + [
    A.HorizontalFlip(p=0.5),               # flip 50%
    A.VerticalFlip(p=0.5),                 # vertical jarang tapi perlu
    A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
    A.HueSaturationValue(5, 10, 10, p=0.3),
    A.Rotate(limit=15, p=0.5),             # rotasi ringan
    A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), shear=5, p=0.5),
    A.GaussNoise(std_range=(0.1, 0.2), p=0.2),
    # A.MotionBlur(blur_limit=5, p=0.2),
    A.CLAHE(p=0.2),                         # bantu objek low-contrast
    A.LongestMaxSize(max_size=TARGET_SIZE),
    A.PadIfNeeded(min_height=TARGET_SIZE, min_width=TARGET_SIZE, border_mode=border_mode)
]

replay_pipeline = ReplayCompose(transform_list)

# =============================================================================
# MEMBACA FILE GAMBAR DAN LABEL
# =============================================================================

# Buat dictionary dengan key berupa nama file tanpa ekstensi
image_files = {os.path.splitext(os.path.basename(f))[0]: f 
               for f in glob(os.path.join(IMAGE_PATH, "*.jpg"))}
label_files = {os.path.splitext(os.path.basename(f))[0]: f 
               for f in glob(os.path.join(LABEL_PATH, "*.txt"))}

# Buat list sorted_keys berdasarkan file asli
sorted_keys = natsorted(list(image_files.keys()))

# Konversi list dan range key ke lowercase untuk perbandingan
sorted_keys_lower = [key.lower() for key in sorted_keys]
start_key = SELECT_IMAGE_START.lower()
end_key = SELECT_IMAGE_END.lower()

try:
    start_idx = sorted_keys_lower.index(start_key)
    end_idx = sorted_keys_lower.index(end_key) + 1
    selected_keys = sorted_keys[start_idx:end_idx]
except ValueError as e:
    print(f"Range tidak ditemukan: pastikan '{SELECT_IMAGE_START}' dan '{SELECT_IMAGE_END}' ada di dalam daftar file.")
    selected_keys = sorted_keys  # Jika range tidak ditemukan, gunakan semua file

# =============================================================================
# PROSES AUGMENTASI SECARA BATCH PER GAMBAR
# =============================================================================

def process_image_batch(filename):
    """
    Proses augmentasi untuk satu gambar beserta label-nya.
    Melakukan augmentasi berulang kali dan menyesuaikan koordinat poligon.
    """
    img_file = image_files.get(filename)
    lbl_file = label_files.get(filename)
    if not img_file or not lbl_file:
        return

    image = cv2.imread(img_file)
    if image is None:
        return
    H, W = image.shape[:2]

    # Baca dan parsing label (format: "class_id x0 y0 x1 y1 ...")
    with open(lbl_file, "r") as f:
        lines = f.readlines()

    polygons = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        class_id = parts[0]
        coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        # Konversi koordinat relatif ke piksel
        coords[:, 0] *= W
        coords[:, 1] *= H
        polygons.append((class_id, coords))

    if not polygons:
        return

    # Lakukan augmentasi NUM_AUGMENTATIONS_PER_IMAGE kali untuk tiap gambar
    for i in range(NUM_AUGMENTATIONS_PER_IMAGE):
        replay_result = replay_pipeline(image=image)
        aug_image = replay_result["image"]
        replay_params = replay_result["replay"]

        new_label_data = []
        for class_id, poly in polygons:
            mask = polygon_to_mask(image.shape, poly)
            # Ubah mask ke format 3 channel
            mask_3c = np.stack([mask] * 3, axis=-1)
            aug_mask_result = A.ReplayCompose.replay(replay_params, image=mask_3c)["image"]
            aug_mask = cv2.cvtColor(aug_mask_result, cv2.COLOR_BGR2GRAY)
            _, aug_mask = cv2.threshold(aug_mask, 127, 255, cv2.THRESH_BINARY)

            contours, _ = cv2.findContours(aug_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            for cnt in contours:
                if cv2.contourArea(cnt) < 200:
                    continue
                epsilon = 0.005 * cv2.arcLength(cnt, True)
                approx = cv2.approxPolyDP(cnt, epsilon, True)
                pts = approx.reshape(-1, 2).astype(float)
            
                pts = ensure_clockwise(pts)
                h2, w2 = aug_image.shape[:2]
                pts[:, 0] /= w2
                pts[:, 1] /= h2
                new_label_data.append((class_id, pts))
        # Hapus label duplikat
        new_label_data = remove_duplicate_labels(new_label_data)

        out_img_path = os.path.join(OUTPUT_IMAGE_PATH, f"{filename}_aug_{i+1}.jpg")
        cv2.imwrite(out_img_path, aug_image)
        out_lbl_path = os.path.join(OUTPUT_LABEL_PATH, f"{filename}_aug_{i+1}.txt")
        with open(out_lbl_path, "w") as f:
            for class_id, pts in new_label_data:
                pts_flat = pts.flatten().tolist()
                f.write(f"{class_id} " + " ".join(map(str, pts_flat)) + "\n")

# =============================================================================
# MENYALIN DATA ASLI (ORIGINAL)
# =============================================================================

def copy_original(filename):
    """
    Menyalin file gambar dan label asli ke direktori output dengan suffix '_ori'.
    """
    img_file = image_files.get(filename)
    lbl_file = label_files.get(filename)
    if img_file:
        image = cv2.imread(img_file)
        if image is not None:
            out_img_path = os.path.join(OUTPUT_IMAGE_PATH, f"{filename}_ori.jpg")
            cv2.imwrite(out_img_path, image)
    if lbl_file:
        out_lbl_path = os.path.join(OUTPUT_LABEL_PATH, f"{filename}_ori.txt")
        with open(lbl_file, "r") as f_in, open(out_lbl_path, "w") as f_out:
            f_out.write(f_in.read())

# =============================================================================
# MAIN FUNCTION
# =============================================================================

def main():
    # Salin data asli jika toggle aktif
    if COPY_ORIGINAL_ENABLED:
        for filename in tqdm(selected_keys, desc="Copying Original Images"):
            copy_original(filename)

    # Proses augmentasi untuk tiap gambar yang telah diseleksi
    for filename in tqdm(selected_keys, desc="Augmenting Images"):
        process_image_batch(filename)
    print("Proses augmentasi selesai!")

if __name__ == "__main__":
    main()

Augmenting Images: 100%|██████████| 89/89 [1:59:35<00:00, 80.63s/it] 

Proses augmentasi selesai!


In [5]:
#-----------------------------------------------------------#
############# STEP 02 : Check hasil Augmentasi ##############
#-----------------------------------------------------------#

import cv2              # Untuk manipulasi gambar
import numpy as np      # Untuk operasi array dan numerik
import random           # Untuk pengacakan (misal: memilih gambar acak, warna)
import os               # Untuk operasi file dan folder
import glob             # Untuk pencarian file dengan pola tertentu
import logging
from tqdm import tqdm   # Untuk progress bar di terminal
import shutil

# ============================================
# Konfigurasi Logging
# ============================================
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.handlers.clear()  # Hapus handler default

# Handler untuk menyimpan log ke file (INFO ke atas)
file_handler = logging.FileHandler("proses.log")
file_handler.setLevel(logging.INFO)
file_formatter = logging.Formatter('%(asctime)s %(levelname)s: %(message)s')
file_handler.setFormatter(file_formatter)
logger.addHandler(file_handler)

# Handler untuk menampilkan log ke terminal (WARNING ke atas)
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.WARNING)
console_formatter = logging.Formatter('%(levelname)s: %(message)s')
console_handler.setFormatter(console_formatter)
logger.addHandler(console_handler)

# ============================================
# Definisi Folder Sumber dan Tujuan
# ============================================
aug_img_dir = r"D:\MyProjects\makanan\Dataset_workshop\other\data\augmented_tempe_image"
aug_label_dir = r"D:\MyProjects\makanan\Dataset_workshop\other\data\augmented_tempe_label"
save_dir = r"D:\MyProjects\makanan\Dataset_workshop\other\data\Cek"

# ============================================
# Pastikan Folder Tujuan Ada dan Kosong
# ============================================
if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)
    logger.info("Folder '%s' dibuat.", save_dir)
else:
    # Kosongkan folder cek jika sudah ada
    for filename in os.listdir(save_dir):
        file_path = os.path.join(save_dir, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.unlink(file_path)
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)
        except Exception as e:
            logger.warning("Gagal menghapus %s. Error: %s", file_path, e)
    logger.info("Folder '%s' dikosongkan.", save_dir)

# ============================================
# Pengambilan File Gambar
# ============================================
img_files = glob.glob(os.path.join(aug_img_dir, "*.jpg"))

if len(img_files) < 100:
    logger.warning("Gambar kurang dari 100! Menampilkan semua yang ada.")
    random_imgs = img_files
else:
    random_imgs = random.sample(img_files, 100)

logger.info("Menampilkan %d gambar untuk pengecekan.", len(random_imgs))

# ============================================
# Fungsi Bantuan
# ============================================

def compute_polygon_area(pts):
    """Menghitung luas poligon menggunakan metode contourArea OpenCV."""
    return cv2.contourArea(pts)

def is_polygon_within_bounds(pts, width, height):
    """Memeriksa apakah semua titik poligon berada dalam batas gambar."""
    for pt in pts.reshape(-1, 2):
        x, y = pt
        if x < 0 or x > width or y < 0 or y > height:
            return False
    return True

def round_polygon(pts, precision=4):
    """Mengembalikan tuple dari koordinat poligon yang sudah dibulatkan."""
    return tuple(np.round(pts.flatten(), precision))

# ============================================
# Proses Pengolahan dan Pengecekan Tiap Gambar
# ============================================
for img_file in tqdm(random_imgs, desc="Memproses gambar"):
    base_name = os.path.splitext(os.path.basename(img_file))[0]
    label_file = os.path.join(aug_label_dir, base_name + ".txt")
    
    if not os.path.exists(label_file):
        logger.warning("Label untuk %s tidak ditemukan, lewati.", base_name)
        continue
    
    image = cv2.imread(img_file)
    if image is None:
        logger.warning("Gagal membaca %s, lewati.", img_file)
        continue
    
    h, w, _ = image.shape
    original_image = image.copy()  # Untuk menggambar anotasi
    
    with open(label_file, "r") as f:
        lines = f.readlines()
    
    # Dictionary untuk mendeteksi duplikasi: key = (class_id, rounded koordinat)
    seen_polygons = {}
    duplicate_found = False
    
    for idx, line in enumerate(lines):
        parts = line.strip().split()
        if len(parts) < 3:
            logger.warning("Format label salah pada %s baris %d.", base_name, idx+1)
            continue

        # Parsing label dan koordinat
        class_id = parts[0]
        try:
            coords = np.array(parts[1:], dtype=np.float32).reshape(-1, 2)
        except Exception as e:
            logger.warning("Gagal parsing koordinat pada %s baris %d. Error: %s", base_name, idx+1, e)
            continue
        
        # Konversi koordinat relatif ke piksel
        coords[:, 0] *= w
        coords[:, 1] *= h
        coords = coords.astype(np.int32)
        pts = coords.reshape((-1, 1, 2))
        
        # Periksa apakah poligon berada dalam batas gambar
        if not is_polygon_within_bounds(pts, w, h):
            logger.warning("Poligon pada %s baris %d berada di luar batas gambar.", base_name, idx+1)
            # Tandai dengan warna oranye
            color = (0, 165, 255)
            cv2.polylines(image, [pts], isClosed=True, color=color, thickness=3)
        else:
            # Tandai dengan warna hijau jika valid
            color = (0, 255, 0)
            cv2.polylines(image, [pts], isClosed=True, color=color, thickness=2)
        
        # Periksa area poligon
        area = compute_polygon_area(pts)
        if area < 10:
            logger.warning("Area poligon terlalu kecil (%.2f) pada %s baris %d.", area, base_name, idx+1)
            # Tandai dengan warna biru
            cv2.polylines(image, [pts], isClosed=True, color=(255, 0, 0), thickness=3)
        
        # Cek duplikasi label
        key = (class_id, round_polygon(pts, precision=2))
        if key in seen_polygons:
            logger.warning("Duplikasi label ditemukan pada %s baris %d. Duplikat dengan baris %d.", 
                           base_name, idx+1, seen_polygons[key])
            duplicate_found = True
            # Tandai duplikasi dengan warna merah
            cv2.polylines(image, [pts], isClosed=True, color=(0, 0, 255), thickness=3)
        else:
            seen_polygons[key] = idx+1  # Simpan nomor baris label
        
    # Simpan gambar hasil pengecekan
    save_path = os.path.join(save_dir, base_name + "_checked.jpg")
    cv2.imwrite(save_path, image)
    
    logger.info("Gambar %s telah dicek dan disimpan di %s", base_name + "_checked.jpg", save_dir)
    
    # Jika ditemukan duplikasi, juga simpan gambar asli untuk referensi
    if duplicate_found:
        dup_save_path = os.path.join(save_dir, base_name + "_duplicate.jpg")
        cv2.imwrite(dup_save_path, original_image)
        logger.info("Gambar asli %s juga disimpan sebagai referensi duplikasi.", base_name)

logger.warning("Proses pengecekan selesai!")

Memproses gambar: 100%|██████████| 100/100 [00:04<00:00, 22.77it/s]


In [ ]:
import os
import shutil
import yaml
import random
from tqdm import tqdm

# -------------------------------
# Konfigurasi jalur direktori
# -------------------------------
raw_dataset_path = r"D:\MyProjects\makanan\Dataset_workshop\data_yolo\Dataset_v6"
output_dataset_path = r"D:\MyProjects\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6"
yaml_output_path = output_dataset_path
yaml_file_path = os.path.join(yaml_output_path, "dataset.yaml")

# Buat struktur folder output
folders = [
    "images/train", "images/val",
    "labels/train", "labels/val"
]
for folder in folders:
    os.makedirs(os.path.join(output_dataset_path, folder), exist_ok=True)

# -------------------------------
# Baca file gambar dan label dari dataset mentah
# -------------------------------
image_dir = os.path.join(raw_dataset_path, "images")
label_dir = os.path.join(raw_dataset_path, "labels")

image_files = sorted([f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
label_files = sorted([f for f in os.listdir(label_dir) if f.lower().endswith('.txt')])

# -------------------------------
# Pisahkan dataset khusus (bawang 410 - 413) dan dataset umum
# -------------------------------
special_dataset = []
regular_dataset = []

for img_file, lbl_file in zip(image_files, label_files):
    if any(f"bawang ({i})" in img_file for i in range(410, 414)):
        special_dataset.append((img_file, lbl_file))
    else:
        regular_dataset.append((img_file, lbl_file))

# -------------------------------
# Split dataset regular (90% train, 10% val)
# -------------------------------
random.shuffle(regular_dataset)
split_idx = int(0.9 * len(regular_dataset))
train_set = regular_dataset[:split_idx]
val_set = regular_dataset[split_idx:]

# -------------------------------
# Split dataset khusus (80% train, 20% val)
# -------------------------------
random.shuffle(special_dataset)
split_idx_special = int(0.8 * len(special_dataset))
train_special = special_dataset[:split_idx_special]
val_special = special_dataset[split_idx_special:]

# -------------------------------
# Gabungkan train dan val
# -------------------------------
train_set += train_special
val_set += val_special

print(f"Total file train: {len(train_set)} (regular + special)")
print(f"Total file val: {len(val_set)} (regular + special)")

# -------------------------------
# Fungsi copy file
# -------------------------------
def copy_file(src, dst):
    shutil.copy(src, dst)

def save_files(entries, folder):
    for img_file, lbl_file in tqdm(entries, desc=f"Menyalin ke {folder}"):
        img_src = os.path.join(image_dir, img_file)
        lbl_src = os.path.join(label_dir, lbl_file)
        img_dst = os.path.join(output_dataset_path, f"images/{folder}", img_file)
        lbl_dst = os.path.join(output_dataset_path, f"labels/{folder}", lbl_file)
        copy_file(img_src, img_dst)
        copy_file(lbl_src, lbl_dst)

save_files(train_set, "train")
save_files(val_set, "val")

# -------------------------------
# Simpan dataset.yaml
# -------------------------------
dataset_yaml = {
    "path": output_dataset_path,
    "train": "images/train",
    "val": "images/val",
    "nc": 2,  # dua kelas: bawang dan tempe
    "names": ["bawang", "tempe"]
}

with open(yaml_file_path, "w") as file:
    yaml.dump(dataset_yaml, file, default_flow_style=False)

print(f"Dataset berhasil di-split dan disimpan di {output_dataset_path}")

In [3]:

#-----------------------------------------------------------#
############# STEP 04 : Check gambar dan label ##############
#-----------------------------------------------------------#

import os

label_path = r"D:\MyProjects\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6\images"
images_path = r"D:\MyProjects\makanan\Dataset_final\Yolo\Dataset_yolo_Segmentation_v6\labels"

labels = set([f.replace('.txt', '') for f in os.listdir(label_path) if f.endswith('.txt')])
images = set([f.replace('.jpg', '').replace('.png', '') for f in os.listdir(images_path) if f.endswith(('.jpg', '.png'))])

missing_labels = images - labels
if missing_labels:
    print("Gambar tanpa label:", missing_labels)
else:
    print("Semua gambar memiliki label.")

Semua gambar memiliki label.


In [ ]:

#-----------------------------------------------------------#
######## STEP 04.1 : (opsional) Cari gambar dan label #######
#-----------------------------------------------------------#

import os
import shutil

# Path asal
label_path = r"D:\MyProjects\makanan\Dataset_yolo_Segmentation_v2\labels\train"
image_path = r"D:\MyProjects\makanan\Dataset_yolo_Segmentation_v2\images\train"
perbaikan_path = r"D:\MyProjects\makanan\testingporkycvv2iyolov8\perbaikan"

# Pastikan folder tujuan ada
os.makedirs(perbaikan_path, exist_ok=True)

# Loop untuk mencari file label kosong dan menyalin gambar yang sesuai
for file in os.listdir(label_path):
    file_path = os.path.join(label_path, file)

    if file.endswith('.txt') and os.path.getsize(file_path) == 0:
        # Salin file label ke folder perbaikan
        shutil.copy(file_path, os.path.join(perbaikan_path, file))
        
        # Cek apakah ada gambar yang sesuai
        base_name = os.path.splitext(file)[0]  # Nama file tanpa ekstensi
        for ext in ['.jpg', '.png', '.jpeg']:  # Cek berbagai format gambar
            image_file = os.path.join(image_path, base_name + ext)
            if os.path.exists(image_file):
                shutil.copy(image_file, os.path.join(perbaikan_path, base_name + ext))
                print(f"File {file} dan gambar {base_name + ext} telah disalin ke folder perbaikan.")

print("Proses selesai.")